In [ ]:
"""
Random Forest Regression (RF)
Compressive Strength Prediction – Gypsum-Based Composites

Author: Haseeb Ahmad

Description:
This script trains a Random Forest model to predict
compressive strength and generates:

    • Training/Test performance metrics
    • Convergence (learning) curve
    • Actual vs Predicted plot (±10% & ±20%)
    • Prediction vs Sample + Error plot
    • Permutation Feature Importance
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance


# ==============================
# Configuration
# ==============================

DATA_PATH = "../data/gypsum_data.csv"
FIGURE_PATH = "../figures/rf_compressive"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_ESTIMATORS = 100

os.makedirs(FIGURE_PATH, exist_ok=True)
np.random.seed(RANDOM_STATE)


# ==============================
# Load Data
# ==============================

def load_data(path):

    df = pd.read_csv(path, header=2)

    X = df.iloc[:, [4, 5, 6, 7, 8, 9, 10]].astype(float)
    y = df.iloc[:, 13].astype(float)   # Compressive strength

    data = pd.concat([X, y], axis=1).dropna()

    X_clean = data.iloc[:, :-1].values
    y_clean = data.iloc[:, -1].values

    return X_clean, y_clean


# ==============================
# Evaluation
# ==============================

def evaluate_model(y_train, y_train_pred, y_test, y_test_pred):

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("=" * 50)
    print("RANDOM FOREST PERFORMANCE")
    print("=" * 50)
    print(f"Training R² : {train_r2:.4f}")
    print(f"Testing R²  : {test_r2:.4f}")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Testing RMSE : {test_rmse:.4f}")
    print(f"Training MAE : {train_mae:.4f}")
    print(f"Testing MAE  : {test_mae:.4f}")
    print("=" * 50)


# ==============================
# Plotting Functions
# ==============================

def plot_convergence(X_train, y_train, X_test, y_test):

    n_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]

    train_scores = []
    test_scores = []

    for n in n_range:

        model = RandomForestRegressor(
            n_estimators=n,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        train_scores.append(r2_score(y_train, model.predict(X_train)))
        test_scores.append(r2_score(y_test, model.predict(X_test)))

    plt.figure(figsize=(5, 4))
    plt.plot(n_range, train_scores, label="Training R²")
    plt.plot(n_range, test_scores, label="Testing R²")

    plt.xlabel("Number of Trees")
    plt.ylabel("R² Score")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/convergence.png", dpi=300)
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred):

    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    plt.figure(figsize=(5, 4))
    plt.scatter(y_true, y_pred, alpha=0.7, edgecolors="k")
    plt.plot(x_vals, x_vals, "r--", label="Perfect Fit")

    plt.fill_between(x_vals, 0.8*x_vals, 1.extrax_vals, alpha=0.15, label="±20%")
    plt.fill_between(x_vals, 0.9*x_vals, 1.1*x_vals, alpha=0.25, label restorative)

    plt.xlabel("Actual Compressive Strength (MPa)")
    plt.ylabel("Predicted Compressive Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/actual_vs_predicted.png", dpi=300)
    plt.show()


def plot_prediction_error(y_true, y_pred):

    indices = np.arange(len(y_true))
    errors = y_pred - y_true

    plt.figure(figsize=(5, 4))
    plt.plot(indices, y_true, "o-", label="Actual")
    plt.plot(indices, y_pred, "x--", label="Predicted")
    plt.plot(indices, errors, ".--", label="Error")

    plt.xlabel("Sample Index")
    plt.ylabel("Compressive Strength (MPa)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/prediction_error.png", dpi=300)
    plt.show()


def plot_feature_importance(model, X_test, y_test, feature_names):

    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=RANDOM_STATE
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURE_PATH}/feature_importance.png", dpi=300)
    plt.show()


# ==============================
# Main
# ==============================

def main():

    # Load data
    X, y = load_data(DATA_PATH)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    # Train model
    rf_model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    # Predictions
    y_train_pred = rf_model.predict(X_train)
    y_test_pred = rf_model.predict(X_test)

    # Evaluate
    evaluate_model(y_train, y_train_pred, y_test, y_test_pred)

    # Plots
    plot_convergence(X_train, y_train, X_test, y_test)
    plot_actual_vs_predicted(y_test, y_test_pred)
    plot_prediction_error(y_test, y_test_pred)

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    plot_feature_importance(
        rf_model,
        X_test,
        y_test,
        feature_names
    )


if __name__ == "__main__":
    main()